# 第 21 课｜规模变大以后，瓶颈会跑到哪里？

一个在 tiny graph 上看起来很均衡的设计，规模变大后可能被 memory、queue，或者少数高流量 neuron 限制。

今天只问一个问题：

> **在某个具体 workload scale 下，怎样判断哪一个 stage 正在限制系统？**

本课主要新概念：**bottleneck 是“相对于 capacity，utilization 最高的 stage”。**

## 1. 概念账本

**已经知道：** latency、throughput、bandwidth、**先进先出队列（First-In First-Out, FIFO）** backpressure、外部 **双倍数据速率（Double Data Rate, DDR）** memory，以及 sparse fan-out。

**今天只学习一个主要概念：** **utilization（利用率）**以及由它识别的、会随 scale 改变的 **bottleneck（瓶颈）**。

**支持术语：** **hotspot** 只表示 traffic/work 集中在少数资源上；本课只要求能识别它可能改变 demand distribution，不要求掌握 bank-conflict 或 cache optimization。

**只预告：** 实测 10K/50K MaleCNS、bank conflict、telemetry，以及 cache / multiple synapse engines 等优化。

## 2. bottleneck 是关系，不是永久标签

对一个 stage，本课采用最小定义：

`utilization = demand / capacity`

utilization 接近或超过 1.0，说明几乎没有 headroom，甚至已经超出能力。

当 graph、event rate、memory pattern 或 architecture 改变时，bottleneck 可以移动。“DDR 永远是瓶颈”不是 specification。

## 3. pipeline 图

<div style="max-width:860px; margin:1rem auto;">
<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 860 220" role="img" aria-label="event pipeline stages" style="width:100%; height:auto; display:block;">
  <defs>
    <marker id="l21-arrow" markerWidth="10" markerHeight="10" refX="9" refY="3" orient="auto">
      <path d="M0,0 L0,6 L9,3 z" fill="#2f5f3f"/>
    </marker>
  </defs>
  <g font-family="sans-serif" font-size="20" text-anchor="middle">
    <rect x="20" y="75" width="145" height="70" rx="8" fill="#e6f4ea" stroke="#3f7a50" stroke-width="2"/><text x="92" y="117" fill="#1f2d24">spike source</text>
    <rect x="190" y="75" width="115" height="70" rx="8" fill="#e6f4ea" stroke="#3f7a50" stroke-width="2"/><text x="247" y="117" fill="#1f2d24">FIFO</text>
    <rect x="330" y="75" width="165" height="70" rx="8" fill="#e6f4ea" stroke="#3f7a50" stroke-width="2"/><text x="412" y="117" fill="#1f2d24">synapse lookup</text>
    <rect x="520" y="75" width="145" height="70" rx="8" fill="#e6f4ea" stroke="#3f7a50" stroke-width="2"/><text x="592" y="105" fill="#1f2d24">DDR /</text><text x="592" y="130" fill="#1f2d24">storage</text>
    <rect x="690" y="75" width="150" height="70" rx="8" fill="#e6f4ea" stroke="#3f7a50" stroke-width="2"/><text x="765" y="117" fill="#1f2d24">target update</text>
  </g>
  <g fill="none" stroke="#2f5f3f" stroke-width="3" marker-end="url(#l21-arrow)">
    <path d="M165 110 L190 110"/><path d="M305 110 L330 110"/><path d="M495 110 L520 110"/><path d="M665 110 L690 110"/>
  </g>
</svg>
</div>

每个 stage 都可能有不同 capacity 与 demand。

## 4. Run：一个 synthetic scale study

下面都是**教学数字**，不是 FPGA 实测结果。目的只是练习分析方法。

In [ ]:
capacities = {
    "fifo": 10.0,
    "lookup": 8.0,
    "memory": 6.0,
    "update": 9.0,
}

cases = {
    "small": {"fifo": 2.0, "lookup": 3.6, "memory": 1.8, "update": 2.2},
    "medium": {"fifo": 5.0, "lookup": 5.5, "memory": 5.7, "update": 4.8},
    "large": {"fifo": 9.5, "lookup": 6.5, "memory": 5.4, "update": 6.8},
}

for name, demand in cases.items():
    utilization = {
        stage: demand[stage] / capacities[stage]
        for stage in capacities
    }
    bottleneck = max(utilization, key=utilization.get)
    print(name, "bottleneck:", bottleneck,
          "utilization:", round(utilization[bottleneck], 2))

## 5. Observe

small case 的最高 utilization 在 lookup；medium case 移到 memory；large case 又移到 FIFO。

这正是本课要观察的事情：bottleneck 是 **demand 相对于 capacity** 的结果，会随 workload scale 与 traffic distribution 改变。这个 toy diagnosis 只告诉我们“先调查哪里”，并不自动证明更深层原因。

## 6. hotspot 是“不均匀的工作”

两个 network 可以拥有相同 edge 总数，却有非常不同的 traffic distribution。少数 high-fanout 或 high-rate source 可能制造 queue pressure 或 bank conflict。

所以 “network size” 不是完整的 performance 描述，distribution 也重要。

## 7. 先测量，再优化

RMD-022 故意排在 correctness baseline 之后。

cache、banking、lazy update 或更多 engine 都不自动等于更好。应该先建立：

1. correct baseline；
2. telemetry 与明确 workload；
3. 实际观察到的 limiting resource；
4. 同 workload 的 before/after benchmark。

## 8. Try It

在 medium case 中只提高 memory capacity。先预测 bottleneck 会移到哪里。

然后在 large case 中只提高 FIFO capacity。新的 highest-utilization stage 是谁？

## 9. 作业

[第 21 课作业：找出 utilization 最高的 stage](../../exercises/zh/21_scaling_bottlenecks.ipynb)

## 10. AI Task

给 AI 一张 utilization table，让它提出 hottest stage 的三个可能原因。要求它把这些写成 hypothesis，而不是伪装成 measurement。

## 11. Human Check

解释为什么 bottleneck 会随 scale 改变。为什么 “raw workload 最大” 不一定等于 “utilization 最高”？选择 optimization 前需要什么证据？

## 12. Engineering Handoff

对应 `RMD-019~022`、`MOD-014 telemetry` 与 `P-001~P-008`。正式性能结论必须来自定义清楚的 workload 与实测 report；本课数字只属于 teaching fixture。

## 13. Project Trace

- Lesson：`LSN-021`
- 映射：`RMD-019 / RMD-020 / RMD-021 / RMD-022`
- 需求路径：`TRACE-P-001`、`TRACE-F-001`
- optimization 前 correctness oracle：`T-016`
- performance metrics：`P-001~P-008`

## 14. Exit Ticket

面对多个 stage 的 demand 与 capacity，你能够计算 utilization、找出当前 bottleneck candidate，并解释为什么换一个 scale 后诊断可能变化。